# M4 · Faster computation: attention, quantization, VRAM

**Question:** Does a more efficient attention kernel beat lower precision? Compare F1 and F2 against the same BF16 40-step baseline, then examine memory saving versus actual latency. Do not multiply independent speedups.

**Learning loop:** predict → inspect frozen evidence → compute → critique → propose a controlled experiment.

Reference lesson: [open course/M04_faster_computation/LESSON.md](../course/M04_faster_computation/LESSON.md). Switch language: [中文](./04_faster_computation.zh-CN.ipynb).

**Default mode:** CPU-only, internet-free, read-only. GPU work is never run implicitly.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## Inspect and recompute real evidence

Run cells in order. Explain every denominator and scope.

In [ ]:
rows = ns.summary_rows()
base = rows["B0"]
for arm in ("F1", "F2"):
    v = rows[arm]
    gain = base["warm_total_median_s"] / v["warm_total_median_s"]
    delta_mb = base["vram_device_peak_mb"] - v["vram_device_peak_mb"]
    print(f"{arm}: speedup={gain:.3f}x, device peak saved={delta_mb:.1f} MB")
plot_values = {name: rows[name]["warm_total_median_s"] for name in ("B0", "F1", "F2")}
plot_ylabel, plot_title = "warm seconds", "Same 40-step workload"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
quality = ns.read_json("evidence/quality/F2/quality.json")
assert quality["semantics"] == "fidelity"
print("F2 LPIPS semantics:", quality["semantics"], "(distance from B0 numerics)")
for p, value in quality["median"].items():
    print(p, f"LPIPS median={value:.4f}")
print("LPIPS is NOT a general image-quality score and does not replace human review.")


## Optional physical GPU smoke — explicit opt-in

Requires the staged ROCm environment and sufficient model/GPU resources. Never used in CPU CI. The run goes to `runs/`, not official evidence.

In [ ]:
RUN_GPU = False  # Only switch on after checking platform quota + available models
if RUN_GPU:
    # Also set QIR_NOTEBOOK_GPU_ACK=YES in your own kernel environment.
    # The helper enforces --smoke and writes under runs/, never evidence/.
    ns.gpu_smoke("F2", "notebook-f2", confirmed=True)
else:
    print("GPU NOT_RUN; evidence analysis above is valid without GPU access.")


## Your turn / Self-check

Calculate the memory savings in GB (decimal) and GiB (binary); do not confuse them. Why might quantization yield only a modest 1.08× speed gain while reducing GPU memory substantially?

**Evidence discipline:** Answer in your own words. This notebook's numbers come from existing committed records, not your GPU. If your optional smoke runs, its data must remain under `runs/` and may not be presented as a full-protocol median.
